In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone https://github.com/dvhanh-code/netiquette-multilabel-classification.git
%cd netiquette-multilabel-classification

Cloning into 'netiquette-multilabel-classification'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 302 (delta 34), reused 66 (delta 17), pack-reused 204 (from 2)
Receiving objects: 100% (302/302), 219.59 MiB | 14.35 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (99/99), done.
/content/netiquette-multilabel-classification


In [ ]:
!pip install transformers datasets accelerate sentencepiece \
              scikit-learn iterative-stratification \
              pandas pyarrow

In [ ]:
!pip install gdown

In [ ]:
DATA_PATH = "/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet"

In [ ]:
import os, shutil, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertModel, BertTokenizer

from src.training.transformer_dataset import (
    NetiquetteTransformerDataset, load_dataset
)
from src.training.transformer_metrics import (
    compute_multilabel_metrics, tune_thresholds, print_metrics_table
)

LOCAL_OUT = "results/gbert_large_gold_silver_128_focal_lr5e6"
DRIVE_OUT = "/content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_focal_lr5e6"
DATA_PATH = "/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet"
MODEL_NAME = "deepset/gbert-large"
MAX_LEN = 128
BATCH = 8
device = torch.device("cuda")

# Copy best model từ Drive
os.makedirs(f"{LOCAL_OUT}/best_model", exist_ok=True)
for fname in os.listdir(f"{DRIVE_OUT}/best_model"):
    shutil.copy2(f"{DRIVE_OUT}/best_model/{fname}",
                 f"{LOCAL_OUT}/best_model/{fname}")
print("Best model restored!")

# Load thresholds từ Drive
shutil.copy2(f"{DRIVE_OUT}/thresholds.json", f"{LOCAL_OUT}/thresholds.json")
best_thresholds = json.load(open(f"{LOCAL_OUT}/thresholds.json"))
print("Thresholds:", best_thresholds)

# Model
class TransformerClassifier(nn.Module):
    def __init__(self, model_name, num_labels=4, dropout=0.2):
        super().__init__()
        self.encoder = BertModel.from_pretrained(
            model_name, ignore_mismatched_sizes=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            self.encoder.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        out = self.encoder(**kw)
        pooled = (out.pooler_output
                  if hasattr(out, "pooler_output") and out.pooler_output is not None
                  else out.last_hidden_state[:, 0])
        return self.classifier(self.dropout(pooled))

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = TransformerClassifier(MODEL_NAME).to(device)
model.load_state_dict(torch.load(
    f"{LOCAL_OUT}/best_model/pytorch_model.bin",
    map_location=device, weights_only=True))
model.eval()

# Data
splits = load_dataset(DATA_PATH, mode="gold_silver")
def make_loader(df):
    return DataLoader(
        NetiquetteTransformerDataset(df, tokenizer, MAX_LEN),
        batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

val_loader  = make_loader(splits["val"])
test_loader = make_loader(splits["test"])

# Collect logits
def collect(loader):
    ll, lb, lm = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch["input_ids"], batch["attention_mask"],
                          batch.get("token_type_ids"))
            ll.append(logits.cpu().numpy())
            lb.append(batch["labels"].cpu().numpy())
            lm.append(batch["label_mask"].cpu().numpy())
    return np.concatenate(ll), np.concatenate(lb), np.concatenate(lm)

print("Running val inference...")
val_logits, val_labels, val_masks = collect(val_loader)

print("Running test inference...")
test_logits, test_labels, test_masks = collect(test_loader)

# Print metrics
test_metrics = compute_multilabel_metrics(
    test_logits, test_labels, test_masks,
    thresholds=best_thresholds, split_name="test")
print_metrics_table("E7 FINAL TEST", test_metrics)

# Save logits
np.savez(f"{LOCAL_OUT}/test_logits.npz",
         logits=test_logits, labels=test_labels, label_mask=test_masks)
np.savez(f"{LOCAL_OUT}/val_logits.npz",
         logits=val_logits, labels=val_labels, label_mask=val_masks)

# Sync to Drive
shutil.copy2(f"{LOCAL_OUT}/test_logits.npz", f"{DRIVE_OUT}/test_logits.npz")
shutil.copy2(f"{LOCAL_OUT}/val_logits.npz",  f"{DRIVE_OUT}/val_logits.npz")
print(" test_logits.npz saved to Drive!")

Best model restored!
Thresholds: {'hate_speech': 0.55, 'toxic': 0.3, 'threat': 0.8, 'insult': 0.45}


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: deepset/gbert-large
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running val inference...
Running test inference...

E7 FINAL TEST
----------------------------------------------------------------------------------------------------
      label  threshold  precision  recall     f1     f2    mcc  s_score  support_pos  support_total
hate_speech     0.5500     0.3227  0.7152 0.4447 0.5752 0.3887   0.6348         1422          13250
      toxic     0.3000     0.5984  0.8315 0.6960 0.7714 0.5499   0.7732          819           2722
     threat     0.8000     0.2500  0.3333 0.2857 0.3125 0.2847   0.4774           21           4376
     insult     0.4500     0.4258  0.7616 0.5462 0.6578 0.4384   0.6885         1296           7098
      MACRO        NaN     0.3992  0.6604 0.4931 0.5792 0.4154   0.6435         3558          27446
 test_logits.npz saved to Drive!
